# duckdb-kql — run KQL queries on DuckDB

[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/mmaitre314/duckdb-kql?devcontainer_path=.devcontainer%2Fdemo%2Fdevcontainer.json)

[`duckdb-kql`](https://github.com/mmaitre314/duckdb-kql) translates
[Kusto Query Language](https://learn.microsoft.com/azure/data-explorer/kusto/query/)
into [DuckDB](https://duckdb.org/) SQL, so you can run KQL against local files and in-process data with
no cluster and no network.

To get started, either open the GitHub Codespace above or install Python followed by `pip install duckdb-kql[all]`.

In [2]:
import random
from datetime import datetime, timedelta

import duckdb_kql

print("duckdb-kql", duckdb_kql.__version__)

duckdb-kql 0.0.1.dev1


## Sample data

Connect to an in-memory instance of DuckDB and populate a sample table `Requests`.

In [3]:
con = duckdb_kql.connect()

start = datetime(2026, 3, 1)

rows = []
for i in range(2_000):
    service = random.choice(["checkout", "search", "auth", "catalog"])
    status = random.choice([200, 200, 200, 500] if service == "auth" else [200, 200, 200, 200, 503])
    rows.append((
        start + timedelta(seconds=i * 37),
        service,
        random.choice(["us-west-2", "us-east-1", "eu-west-1"]),
        status,
        round(random.lognormvariate(3.2, 0.6), 1),
        f"user-{random.randint(1, 120):03d}",
    ))

con.execute("CREATE TABLE Requests (Timestamp TIMESTAMP, Service VARCHAR, Region VARCHAR, Status INTEGER, LatencyMs DOUBLE, UserId VARCHAR)")
con.executemany("INSERT INTO Requests VALUES (?, ?, ?, ?, ?, ?)", rows)

print(duckdb_kql.engine.schema(con))

{'Requests': ['Timestamp', 'Service', 'Region', 'Status', 'LatencyMs', 'UserId']}


## Layer 0 — KQL in, SQL out

No database, no connection, no data. This layer is pure text-to-text, and it is
the one you want if you are generating SQL to run somewhere else.

In [2]:
kql = """
Requests
| where Status >= 500 and Region has "west"
| summarize Failures = count(), Users = dcount(UserId) by Service
| sort by Failures desc
| take 3
"""

print(duckdb_kql.to_sql(kql))

WITH _s0 AS (SELECT * FROM "Requests"),
     _s1 AS (SELECT * FROM _s0 WHERE (("Status" >= CAST(500 AS BIGINT)) AND coalesce(regexp_matches("Region", '(?i)\b' || regexp_escape('west') || '\b'), FALSE))),
     _s2 AS (SELECT "Service" AS "Service", count(*) AS "Failures", count(DISTINCT "UserId") AS "Users" FROM _s1 GROUP BY "Service"),
     _s3 AS (SELECT * FROM _s2 ORDER BY "Failures" DESC NULLS LAST),
     _s4 AS (SELECT * FROM _s3 LIMIT 3)
SELECT * FROM _s4


Note the shape: **one CTE per KQL operator**, in pipeline order. DuckDB's
optimizer collapses the chain, so there is no cost to it, and it means you can
read the generated SQL against the query you wrote. That matters when you are
debugging a result you did not expect.

Two other Layer 0 entry points — checking a query without running it, and
finding out what parameters it declares:

In [3]:
# validate() reports syntax problems with source spans and never raises.
for diagnostic in duckdb_kql.validate("Requests | wehre Status == 500"):
    print(diagnostic)

print()
print("well-formed query ->", duckdb_kql.validate("Requests | take 1"))

1:9: mismatched input '|' expecting {<EOF>, ';'}

well-formed query -> []


### Refusing is a feature

An unsupported construct raises with the construct's name. It never falls back
to something *close enough* — that is precisely how a plausible wrong answer
gets shipped.

In [4]:
for query in [
    "Requests | evaluate bag_unpack(Payload)",
    "Requests | summarize hll(UserId)",
]:
    try:
        duckdb_kql.to_sql(query)
    except duckdb_kql.KqlError as exc:
        print(f"{type(exc).__name__}: {exc}")

KqlUnsupportedError: unsupported KQL construct 'EvaluateOperator' at 1:11 (not implemented in this wave; near 'evaluatebag_unpack(Payload)')
KqlUnsupportedError: unsupported KQL construct 'aggregate:hll' (no DuckDB mapping in this wave; see translate/functions.py)


## Layer 1 — actually run it

`duckdb_kql.connect()` is `duckdb.connect()` plus the `TimeZone='UTC'` that KQL
datetime semantics require. Everything else is a normal DuckDB connection, so
your own tables, Parquet files and CSVs are all queryable.

{'Requests': ['Timestamp', 'Service', 'Region', 'Status', 'LatencyMs', 'UserId']}


In [6]:
# sql() returns a DuckDB relation — lazy, composable, and printable.
duckdb_kql.sql(con, """
Requests
| where Status >= 500
| summarize Failures = count(), Users = dcount(UserId) by Service, Region
| sort by Failures desc
| take 5
""")

┌──────────┬───────────┬──────────┬───────┐
│ Service  │  Region   │ Failures │ Users │
│ varchar  │  varchar  │  int64   │ int64 │
├──────────┼───────────┼──────────┼───────┤
│ checkout │ us-east-1 │       42 │    35 │
│ catalog  │ us-west-2 │       39 │    34 │
│ search   │ us-east-1 │       38 │    34 │
│ checkout │ eu-west-1 │       38 │    35 │
│ checkout │ us-west-2 │       38 │    30 │
└──────────┴───────────┴──────────┴───────┘

`df()` gives you a pandas DataFrame, and `arrow()` a PyArrow table:

In [7]:
frame = duckdb_kql.df(con, """
Requests
| summarize
    Requests = count(),
    P50 = percentile(LatencyMs, 50),
    P95 = percentile(LatencyMs, 95)
  by Service
| sort by P95 desc
""")
frame

,Service,Requests,P50,P95
0,auth,451,24.8,66.6
1,search,524,24.1,65.0
2,catalog,487,24.6,63.8
3,checkout,538,25.5,63.6


Time bucketing works the way KQL spells it, with `bin()` and timespan literals:

In [8]:
duckdb_kql.df(con, """
Requests
| where Status >= 500
| summarize Errors = count() by bin(Timestamp, 4h)
| sort by Timestamp asc
""")

,Timestamp,Errors
0,2026-03-01 00:00:00,85
1,2026-03-01 04:00:00,92
2,2026-03-01 08:00:00,76
3,2026-03-01 12:00:00,79
4,2026-03-01 16:00:00,81
5,2026-03-01 20:00:00,11


## Parameters, and why they are not string formatting

`declare query_parameters` values are bound through DuckDB's parameter API. The
generated SQL contains a placeholder — **not the value, and not even the
parameter's name**. There is nothing to escape because nothing is interpolated.

In [9]:
parameterized = """
declare query_parameters(service:string, floor:long);
Requests
| where Service == service and Status >= floor
| count
"""

translated = duckdb_kql.to_sql(parameterized, parameters={"service": "auth", "floor": 500})
print(translated)
print()
print("values travel beside the SQL:", translated.parameters)

WITH _s0 AS (SELECT * FROM "Requests"),
     _s1 AS (SELECT * FROM _s0 WHERE (CASE WHEN "Service" IS NULL AND CAST($kqlp0 AS VARCHAR) IS NULL THEN NULL ELSE coalesce(("Service" = CAST($kqlp0 AS VARCHAR)), FALSE) END AND ("Status" >= CAST($kqlp1 AS BIGINT)))),
     _s2 AS (SELECT count(*) AS "Count" FROM _s1)
SELECT * FROM _s2

values travel beside the SQL: {'kqlp0': 'auth', 'kqlp1': 500}


In [10]:
# The point of that design, demonstrated. This payload is a valid KQL fragment
# and a classic SQL injection attempt; it is neither here, because it never
# reaches the SQL text at all — it is only ever compared as a string.
payload = "auth' OR 1=1 --"

print("rows for a real service :", duckdb_kql.sql(con, parameterized,
      {"service": "auth", "floor": 500}).fetchall())
print("rows for the payload    :", duckdb_kql.sql(con, parameterized,
      {"service": payload, "floor": 500}).fetchall())

sql_text = str(duckdb_kql.to_sql(parameterized, parameters={"service": payload, "floor": 500}))
assert payload not in sql_text
print("\npayload present in generated SQL:", payload in sql_text)

rows for a real service : [(101,)]
rows for the payload    : [(0,)]

payload present in generated SQL: False


You can also ask a query what it wants before supplying anything:

In [11]:
for declaration in duckdb_kql.query_parameters(parameterized):
    print(f"{declaration.name:>8} : {declaration.type}")

 service : string
   floor : long


## Layer 2 — the `azure-kusto-data` shape

If you already have code written against the Kusto Python SDK, this layer lets
it run against DuckDB with the import swapped. The response objects match the
SDK's attribute for attribute, and `raw_rows` carries Kusto's *wire* format, so
the SDK's own converters work on them unchanged.

In [12]:
from duckdb_kql.kusto import KustoClient
from duckdb_kql.kusto.helpers import dataframe_from_result_table

client = KustoClient(con)

response = client.execute("NetDefaultDB", """
Requests
| where Status >= 500
| summarize Errors = count() by Service
| sort by Errors desc
""")

table = response.primary_results[0]
print("columns  :", [(c.column_name, c.column_type) for c in table.columns])
print("raw_rows :", table.raw_rows[:2])
print()
dataframe_from_result_table(table)

columns  : [('Service', 'string'), ('Errors', 'long')]
raw_rows : [['checkout', 118], ['catalog', 104]]



,Service,Errors
0,checkout,118
1,catalog,104
2,search,101
3,auth,101


Parameters go through `ClientRequestProperties`, exactly as they do against a
real cluster:

In [13]:
from duckdb_kql.kusto import ClientRequestProperties

properties = ClientRequestProperties()
properties.set_parameter("service", "auth")

result = client.execute(
    "NetDefaultDB",
    "declare query_parameters(service:string);\nRequests | where Service == service | count",
    properties,
)
print(result.primary_results[0].rows[0]["Count"])

451


### Options are implemented or refused — never ignored

A request option that silently did nothing would be the worst kind of
compatibility: your code keeps running and stops doing what it says. Every
`ClientRequestProperties` option is classified, and the ones that cannot be
honoured raise.

In [14]:
from duckdb_kql.kusto.exceptions import KustoError

properties = ClientRequestProperties()
properties.set_option(ClientRequestProperties.request_timeout_option_name, dt.timedelta(seconds=30))
print("servertimeout: accepted and enforced")

try:
    properties.set_option("query_results_cache_max_age", dt.timedelta(minutes=5))
except KustoError as exc:
    print(f"{type(exc).__name__}: {exc}")

servertimeout: accepted and enforced
KustoUnsupportedError: unsupported by duckdb-kql: request option 'query_results_cache_max_age' (There is no results cache, so a max age would govern nothing.)


## The traps this project exists for

KQL and SQL share an enormous amount of syntax and disagree in small, quiet
places. Each of the following is verified against the real Kusto engine (the
Kusto Emulator) rather than inferred from documentation — several of them
contradict the documentation.

### `has` is term-based; `contains` is substring

`Region has "west"` is **false** for `"westward"`. Translating `has` as
`LIKE '%west%'` would return extra rows on real log data, and nothing would
look wrong.

In [15]:
duckdb_kql.df(con, """
datatable(Text:string) ["west", "westward", "us-west-2", "WEST"]
| extend
    has_west = Text has "west",
    contains_west = Text contains "west",
    has_cs_west = Text has_cs "west"
""")

,Text,has_west,contains_west,has_cs_west
0,west,True,True,True
1,westward,False,True,False
2,us-west-2,True,True,True
3,WEST,True,True,False


### Null is the *smallest* value when sorting

Ascending puts nulls first; descending puts them last. DuckDB's own default is
`NULLS LAST` in both directions.

In [16]:
duckdb_kql.sql(con, "datatable(x:int) [3, int(null), 1] | sort by x asc").fetchall()

[(None,), (1,), (3,)]

### Negated operators keep null rows

`s !contains "x"` is **true** when `s` is null — a plain `NOT (...)` in SQL
yields `NULL`, and `where` then drops the row. That turns into a count that is
quietly too low.

In [17]:
con.execute("CREATE OR REPLACE TABLE Notes(rid INTEGER, note VARCHAR)")
con.executemany("INSERT INTO Notes VALUES (?, ?)", [(1, "timeout"), (2, None)])

kept = duckdb_kql.sql(con, 'Notes | where note !contains "retry" | count').fetchone()[0]
naive = con.sql("SELECT count(*) FROM Notes WHERE NOT (note ILIKE '%retry%')").fetchone()[0]

print(f"KQL semantics           : {kept} rows")
print(f"a naive NOT (...) in SQL: {naive} rows   <- the null row silently vanishes")
print()
print("Neither note mentions 'retry', so both qualify. Nothing errors either way;")
print("the naive translation just returns a smaller number.")

KQL semantics           : 2 rows
a naive NOT (...) in SQL: 1 rows   <- the null row silently vanishes

Neither note mentions 'retry', so both qualify. Nothing errors either way;
the naive translation just returns a smaller number.


### `join` defaults to `innerunique`, not SQL's inner join

This is the most dangerous default in the language: a bare `join` deduplicates
the **left** side's join keys first. Two rows below, where SQL's inner join
gives three — and on real data that difference shows up as inflated counts and
sums that nobody thinks to question.

In [18]:
duckdb_kql.sql(con, """
let L = datatable(k:string, v:int) ["a", 1, "a", 2, "b", 3];
let R = datatable(k:string, w:int) ["a", 10, "b", 20];
L | join R on k
| sort by k asc
""").fetchall()

[('a', 1, 'a', 10), ('b', 3, 'b', 20)]

Ask for SQL's semantics and you get them — but you have to ask:

In [19]:
duckdb_kql.sql(con, """
let L = datatable(k:string, v:int) ["a", 1, "a", 2, "b", 3];
let R = datatable(k:string, w:int) ["a", 10, "b", 20];
L | join kind=inner R on k
| sort by k asc
""").fetchall()

[('a', 1, 'a', 10), ('a', 2, 'a', 10), ('b', 3, 'b', 20)]

## Translating at build time

There is also a `duckdb-kql` command that turns `.kql` files into `.sql` files.
The point is that the *output* has no dependency on this package: translate in
CI, ship the SQL, and run it with nothing but a DuckDB driver.

In [20]:
import tempfile

workdir = Path(tempfile.mkdtemp())
(workdir / "queries").mkdir()
(workdir / "queries" / "failures.kql").write_text(
    'Requests\n| where Status >= 500\n| summarize Errors = count() by Service\n'
)

# The trailing separator on -o means "a directory of outputs". Without it, a
# single input is written to exactly that path, as a file.
outdir = f"{workdir / 'sql'}/"
subprocess.run(
    [sys.executable, "-m", "duckdb_kql", str(workdir / "queries"), "-o", outdir],
    check=True,
)

print((workdir / "sql" / "failures.sql").read_text())

-- Generated by duckdb-kql from /tmp/tmp0k8poo86/queries/failures.kql. Do not edit.
--
-- Run with TimeZone set to UTC:  SET TimeZone='UTC';
-- KQL datetimes are UTC, and DuckDB reads the session zone when casting
-- text without an offset. Without it, datetimes are silently shifted
-- rather than rejected.

WITH _s0 AS (SELECT * FROM "Requests"),
     _s1 AS (SELECT * FROM _s0 WHERE ("Status" >= CAST(500 AS BIGINT))),
     _s2 AS (SELECT "Service" AS "Service", count(*) AS "Errors" FROM _s1 GROUP BY "Service")
SELECT * FROM _s2



The header is not decoration: `SET TimeZone='UTC'` is a *requirement* of the
generated SQL — KQL datetimes are UTC, and DuckDB reads the session timezone
when casting — and nothing else would tell whoever runs the file about it.

`--check` fails a build when a generated `.sql` has fallen behind its `.kql`,
which is what makes this safe to commit and rely on:

In [21]:
# Edit the query, leave the generated SQL alone, and the check fails.
(workdir / "queries" / "failures.kql").write_text(
    'Requests\n| where Status >= 500\n| summarize Errors = count() by Service\n| take 10\n'
)

stale = subprocess.run(
    [sys.executable, "-m", "duckdb_kql", str(workdir / "queries"), "-o", outdir, "--check"],
    capture_output=True,
    text=True,
)
print("exit code:", stale.returncode)
print(stale.stdout.strip() or stale.stderr.strip())

exit code: 3
duckdb-kql: 1 generated file(s) are missing or out of date:
  /tmp/tmp0k8poo86/sql/failures.sql
Re-run without --check to regenerate.


## Where to go next

- **[KQL support matrix](https://github.com/mmaitre314/duckdb-kql/blob/main/docs/kql-support.md)**
  — every operator and function, supported or not, each with its known
  limitations and divergences. Generated from the translator's registries, so it
  cannot claim support that does not exist.
- **[Getting started](https://github.com/mmaitre314/duckdb-kql/blob/main/docs/getting-started.md)**
  and the **[API reference](https://github.com/mmaitre314/duckdb-kql/blob/main/docs/api.md)**.
- **[Kusto SDK compatibility](https://github.com/mmaitre314/duckdb-kql/blob/main/docs/kusto-client.md)**
  — what Layer 2 implements, what it refuses, and why.
- **[Build-time translation](https://github.com/mmaitre314/duckdb-kql/blob/main/docs/cli.md)**.

Found a query that runs and returns something different from Kusto? That is the
most valuable bug report this project can get —
[open an issue](https://github.com/mmaitre314/duckdb-kql/issues).